In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:

import pandas as pd
import requests

from datetime import datetime, timezone
from zoneinfo import ZoneInfo

PROJETO = "macrodata-brasil"
DATA_INICIAL_BCB = "01/01/2024"
PERIODO_INICIAL_IPCA = "202311"

HOJE = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).date()

URL_BCB = "https://api.bcb.gov.br/dados/serie/bcdata.sgs"
URL_IPCA = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/63"


def extrair_bcb(codigo, fonte):
    url = f"{URL_BCB}.{codigo}/dados"

    params = {
        "formato": "json",
        "dataInicial": DATA_INICIAL_BCB,
        "dataFinal": HOJE.strftime("%d/%m/%Y"),
    }

    resposta = requests.get(
        url,
        params=params,
        timeout=60,
    )
    resposta.raise_for_status()

    df = pd.DataFrame(resposta.json())

    if df.empty:
        raise ValueError(f"{fonte}: extração vazia")

    datas = pd.to_datetime(
        df["data"],
        format="%d/%m/%Y",
        errors="coerce",
    )
    valores = pd.to_numeric(
        df["valor"],
        errors="coerce",
    )

    if (
        datas.isna().any()
        or valores.isna().any()
        or (valores <= 0).any()
        or datas.duplicated().any()
    ):
        raise ValueError(
            f"{fonte}: dados inválidos ou duplicados"
        )

    df["data_extracao"] = datetime.now(timezone.utc)
    df["fonte"] = fonte

    return df[
        ["data", "valor", "data_extracao", "fonte"]
    ].copy()


def extrair_ipca_historico():
    url = f"{URL_IPCA}/p/all"

    resposta = requests.get(
        url,
        timeout=120,
    )
    resposta.raise_for_status()

    dados = resposta.json()

    if not isinstance(dados, list) or len(dados) < 2:
        raise ValueError("IPCA: resposta vazia")

    mapeamento = {
        "D1N": "localidade",
        "D2C": "codigo_indicador",
        "D2N": "indicador",
        "D3C": "periodo",
        "D3N": "descricao_periodo",
        "V": "valor",
        "MN": "unidade",
    }

    df = pd.DataFrame(dados[1:])

    if not set(mapeamento).issubset(df.columns):
        raise ValueError("IPCA: estrutura inesperada")

    df = df[list(mapeamento)].rename(
        columns=mapeamento
    )

    df = df[
        df["periodo"] >= PERIODO_INICIAL_IPCA
    ].copy()

    df = df[
        (df["codigo_indicador"] == "63")
        & (df["localidade"] == "Brasil")
    ].copy()

    periodos = pd.to_datetime(
        df["periodo"],
        format="%Y%m",
        errors="coerce",
    )

    valores = pd.to_numeric(
        df["valor"],
        errors="coerce",
    )

    if (
        df.empty
        or periodos.isna().any()
        or valores.isna().any()
        or df["periodo"].duplicated().any()
    ):
        raise ValueError(
            "IPCA: dados inválidos ou duplicados"
        )

    for coluna in mapeamento.values():
        df[coluna] = df[coluna].astype(str)

    df["data_extracao"] = datetime.now(timezone.utc)
    df["fonte"] = "IBGE_SIDRA_1737_63"

    return df.sort_values(
        "periodo"
    ).reset_index(drop=True)


df_selic = extrair_bcb(432, "BCB_SGS_432")
df_cambio = extrair_bcb(1, "BCB_SGS_1")
df_ipca = extrair_ipca_historico()

print("Reprocessamento histórico — Validação inicial")
print()

for nome, df, coluna in [
    ("Selic", df_selic, "data"),
    ("Câmbio", df_cambio, "data"),
    ("IPCA", df_ipca, "periodo"),
]:
    if coluna == "data":
        datas = pd.to_datetime(
            df[coluna],
            format="%d/%m/%Y",
        )
    else:
        datas = pd.to_datetime(
            df[coluna],
            format="%Y%m",
        )

    print(
        f"{nome}: {len(df)} registros | "
        f"{datas.min().date()} até "
        f"{datas.max().date()}"
    )

print()
print("Histórico extraído e validado.")
print("Nenhuma tabela do BigQuery foi modificada.")


Reprocessamento histórico — Validação inicial

Selic: 1052 registros | 2024-01-01 até 2026-11-17
Câmbio: 723 registros | 2024-01-02 até 2026-11-16
IPCA: 36 registros | 2023-11-01 até 2026-10-01

Histórico extraído e validado.
Nenhuma tabela do BigQuery foi modificada.


In [4]:

import io

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(
    project=PROJETO,
    location="southamerica-east1",
)

tabelas_historicas = {
    "selic": f"{PROJETO}.bronze.bcb_selic_historico_staging",
    "cambio": f"{PROJETO}.bronze.bcb_cambio_historico_staging",
    "ipca": f"{PROJETO}.bronze.ibge_ipca_historico_staging",
}

dados_historicos = {
    "selic": df_selic,
    "cambio": df_cambio,
    "ipca": df_ipca,
}


def carregar_historico(fonte, df):
    tabela = tabelas_historicas[fonte]

    if fonte == "ipca":
        colunas = [
            "localidade",
            "codigo_indicador",
            "indicador",
            "periodo",
            "descricao_periodo",
            "valor",
            "unidade",
            "data_extracao",
            "fonte",
        ]

        chaves = [
            "localidade",
            "codigo_indicador",
            "periodo",
        ]
    else:
        colunas = [
            "data",
            "valor",
            "data_extracao",
            "fonte",
        ]

        chaves = ["data"]

    if df.empty or not set(colunas).issubset(df.columns):
        raise ValueError(f"{fonte}: dados incompletos")

    if df.duplicated(subset=chaves).any():
        raise ValueError(f"{fonte}: chaves duplicadas")

    schema = [
        bigquery.SchemaField(
            coluna,
            "TIMESTAMP" if coluna == "data_extracao" else "STRING",
        )
        for coluna in colunas
    ]

    jsonl = df[colunas].to_json(
        orient="records",
        lines=True,
        date_format="iso",
        date_unit="ms",
        force_ascii=False,
    )

    arquivo = io.BytesIO(jsonl.encode("utf-8"))

    configuracao = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        schema=schema,
    )

    job = client.load_table_from_file(
        arquivo,
        destination=tabela,
        job_config=configuracao,
        location="southamerica-east1",
    )

    job.result()

    chave_sql = ", ".join(chaves)

    consulta = f"""
        SELECT
            (SELECT COUNT(*)
             FROM `{tabela}`) AS total,

            (SELECT COUNT(*)
             FROM (
                 SELECT {chave_sql}
                 FROM `{tabela}`
                 GROUP BY {chave_sql}
             )) AS chaves_unicas
    """

    resultado = next(iter(
        client.query(
            consulta,
            location="southamerica-east1",
        ).result()
    ))

    if (
        resultado.total != len(df)
        or resultado.chaves_unicas != resultado.total
    ):
        raise RuntimeError(
            f"{fonte}: divergência na carga histórica. "
            f"Extraídos: {len(df)}, "
            f"gravados: {resultado.total}, "
            f"chaves únicas: {resultado.chaves_unicas}"
        )

    print(
        f"{fonte.upper()}: {resultado.total} registros "
        f"validados em {tabela}"
    )


for fonte, df in dados_historicos.items():
    carregar_historico(fonte, df)

print()
print("Carga histórica concluída.")
print("Tabelas Bronze oficiais preservadas.")


SELIC: 1052 registros validados em macrodata-brasil.bronze.bcb_selic_historico_staging
CAMBIO: 723 registros validados em macrodata-brasil.bronze.bcb_cambio_historico_staging
IPCA: 36 registros validados em macrodata-brasil.bronze.ibge_ipca_historico_staging

Carga histórica concluída.
Tabelas Bronze oficiais preservadas.


In [5]:

from google.api_core.exceptions import NotFound

fontes = {
    "selic": {
        "historico": tabelas_historicas["selic"],
        "bronze": f"{PROJETO}.bronze.bcb_selic_raw",
        "chave": "data",
        "inicio": "01/01/2024",
    },
    "cambio": {
        "historico": tabelas_historicas["cambio"],
        "bronze": f"{PROJETO}.bronze.bcb_cambio_raw",
        "chave": "data",
        "inicio": "02/01/2024",
    },
    "ipca": {
        "historico": tabelas_historicas["ipca"],
        "bronze": f"{PROJETO}.bronze.ibge_ipca_raw",
        "chave": "periodo",
        "inicio": "202311",
    },
}


def consultar(sql):
    return next(iter(
        client.query(
            sql,
            location="southamerica-east1",
        ).result()
    ))


def auditar_historico(fonte, config):
    historico = config["historico"]
    bronze = config["bronze"]
    chave = config["chave"]

    if fonte == "ipca":
        expressao_data = (
            "SAFE.PARSE_DATE('%Y%m', periodo)"
        )
        colunas = [
            "localidade",
            "codigo_indicador",
            "indicador",
            "periodo",
            "descricao_periodo",
            "valor",
            "unidade",
            "fonte",
        ]
    else:
        expressao_data = (
            "SAFE.PARSE_DATE('%d/%m/%Y', data)"
        )
        colunas = ["data", "valor", "fonte"]

    resultado = consultar(f"""
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT {chave}) AS chaves_unicas,
            COUNTIF(
                {expressao_data} IS NULL
                OR SAFE_CAST(valor AS NUMERIC) IS NULL
            ) AS invalidos,
            MIN({expressao_data}) AS primeira_data,
            MAX({expressao_data}) AS ultima_data
        FROM `{historico}`
    """)

    if (
        resultado.total == 0
        or resultado.total != resultado.chaves_unicas
        or resultado.invalidos > 0
    ):
        raise RuntimeError(
            f"{fonte}: histórico incompleto ou inválido"
        )

    inicio_esperado = pd.to_datetime(
        config["inicio"],
        format="%Y%m" if fonte == "ipca"
        else "%d/%m/%Y",
    ).date()

    if resultado.primeira_data != inicio_esperado:
        raise RuntimeError(
            f"{fonte}: início inesperado. "
            f"Esperado: {inicio_esperado}, "
            f"encontrado: {resultado.primeira_data}"
        )

    print(
        f"{fonte.upper()}: {resultado.total} registros | "
        f"{resultado.primeira_data} até "
        f"{resultado.ultima_data}"
    )

    try:
        client.get_table(bronze)
    except NotFound:
        print(
            "  Bronze oficial não encontrada. "
            "Reconstrução será necessária."
        )
        return

    campos = ", ".join(colunas)

    diferencas = consultar(f"""
        SELECT
            (SELECT COUNT(*) FROM (
                SELECT {campos} FROM `{historico}`
                EXCEPT DISTINCT
                SELECT {campos} FROM `{bronze}`
            )) AS somente_historico,

            (SELECT COUNT(*) FROM (
                SELECT {campos} FROM `{bronze}`
                EXCEPT DISTINCT
                SELECT {campos} FROM `{historico}`
            )) AS somente_bronze
    """)

    print(
        "  Diferenças entre histórico e Bronze: "
        f"{diferencas.somente_historico} no histórico, "
        f"{diferencas.somente_bronze} na Bronze"
    )


for fonte, config in fontes.items():
    auditar_historico(fonte, config)

print()
print("Auditoria histórica concluída.")
print("Nenhuma tabela foi modificada.")


SELIC: 1052 registros | 2024-01-01 até 2026-11-17
  Diferenças entre histórico e Bronze: 0 no histórico, 0 na Bronze
CAMBIO: 723 registros | 2024-01-02 até 2026-11-16
  Diferenças entre histórico e Bronze: 0 no histórico, 0 na Bronze
IPCA: 36 registros | 2023-11-01 até 2026-10-01
  Diferenças entre histórico e Bronze: 31 no histórico, 31 na Bronze

Auditoria histórica concluída.
Nenhuma tabela foi modificada.


In [7]:

fontes_validacao = {
    "selic": {
        "tabela": tabelas_historicas["selic"],
        "chave": "data",
        "inicio": "2024-01-01",
    },
    "cambio": {
        "tabela": tabelas_historicas["cambio"],
        "chave": "data",
        "inicio": "2024-01-02",
    },
    "ipca": {
        "tabela": tabelas_historicas["ipca"],
        "chave": "periodo",
        "inicio": "2023-11-01",
    },
}


def validar_reconstrucao(fonte, config):
    tabela = config["tabela"]
    chave = config["chave"]

    formato = (
        "%Y%m" if fonte == "ipca"
        else "%d/%m/%Y"
    )

    expressao = (
        f"SAFE.PARSE_DATE('{formato}', {chave})"
    )

    sql = f"""
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT {chave}) AS chaves_unicas,
            COUNTIF(
                {expressao} IS NULL
                OR SAFE_CAST(valor AS NUMERIC) IS NULL
            ) AS invalidos,
            MIN({expressao}) AS primeira_data,
            MAX({expressao}) AS ultima_data
        FROM `{tabela}`
    """

    resultado = next(iter(
        client.query(
            sql,
            location="southamerica-east1",
        ).result()
    ))

    if resultado.total == 0:
        raise RuntimeError(
            f"{fonte}: histórico vazio"
        )

    if resultado.total != resultado.chaves_unicas:
        raise RuntimeError(
            f"{fonte}: existem registros duplicados"
        )

    if resultado.invalidos > 0:
        raise RuntimeError(
            f"{fonte}: existem registros inválidos"
        )

    if str(resultado.primeira_data) != config["inicio"]:
        raise RuntimeError(
            f"{fonte}: histórico começa em "
            f"{resultado.primeira_data}, "
            f"esperado {config['inicio']}"
        )

    print(
        f"{fonte.upper()}: APROVADO | "
        f"{resultado.total} registros | "
        f"{resultado.primeira_data} até "
        f"{resultado.ultima_data}"
    )


for fonte, config in fontes_validacao.items():
    validar_reconstrucao(fonte, config)

print()
print("Histórico aprovado para reconstrução.")
print("Nenhuma tabela oficial foi modificada.")


SELIC: APROVADO | 1052 registros | 2024-01-01 até 2026-11-17
CAMBIO: APROVADO | 723 registros | 2024-01-02 até 2026-11-16
IPCA: APROVADO | 36 registros | 2023-11-01 até 2026-10-01

Histórico aprovado para reconstrução.
Nenhuma tabela oficial foi modificada.


In [ ]:

from datetime import datetime, timezone

from google.api_core.exceptions import NotFound
from google.cloud import bigquery

EXECUTAR_RECONSTRUCAO = True

tabelas_reconstrucao = {
    "selic": {
        "origem": tabelas_historicas["selic"],
        "destino": f"{PROJETO}.bronze.bcb_selic_raw",
        "chaves": ["data"],
        "inicio": "2024-01-01",
        "formato": "%d/%m/%Y",
    },
    "cambio": {
        "origem": tabelas_historicas["cambio"],
        "destino": f"{PROJETO}.bronze.bcb_cambio_raw",
        "chaves": ["data"],
        "inicio": "2024-01-02",
        "formato": "%d/%m/%Y",
    },
    "ipca": {
        "origem": tabelas_historicas["ipca"],
        "destino": f"{PROJETO}.bronze.ibge_ipca_raw",
        "chaves": [
            "localidade",
            "codigo_indicador",
            "periodo",
        ],
        "inicio": "2023-11-01",
        "formato": "%Y%m",
    },
}


def consultar_resultado(sql):
    return next(iter(
        client.query(
            sql,
            location="southamerica-east1",
        ).result()
    ))


def contar_linhas(tabela):
    return consultar_resultado(
        f"SELECT COUNT(*) AS total FROM `{tabela}`"
    ).total


def validar_tabela_historica(fonte, config):
    origem = config["origem"]
    chaves = ", ".join(config["chaves"])
    coluna_data = config["chaves"][-1]

    expressao_data = (
        f"SAFE.PARSE_DATE("
        f"'{config['formato']}', {coluna_data})"
    )

    resultado = consultar_resultado(f"""
        SELECT
            COUNT(*) AS total,
            COUNTIF(
                {expressao_data} IS NULL
                OR SAFE_CAST(valor AS NUMERIC) IS NULL
            ) AS invalidos,
            MIN({expressao_data}) AS inicio,
            COUNT(*) - (
                SELECT COUNT(*)
                FROM (
                    SELECT {chaves}
                    FROM `{origem}`
                    GROUP BY {chaves}
                )
            ) AS duplicados
        FROM `{origem}`
    """)

    if (
        resultado.total == 0
        or resultado.invalidos > 0
        or resultado.duplicados > 0
        or str(resultado.inicio) != config["inicio"]
    ):
        raise RuntimeError(
            f"{fonte}: histórico não passou na validação"
        )

    return resultado.total


def comparar_tabelas(origem, destino):
    consulta = f"""
        SELECT
            (SELECT COUNT(*) FROM `{origem}`) AS total_origem,
            (SELECT COUNT(*) FROM `{destino}`) AS total_destino,
            (
                SELECT COUNT(*)
                FROM (
                    SELECT * FROM `{origem}`
                    EXCEPT DISTINCT
                    SELECT * FROM `{destino}`
                )
            ) AS ausentes_destino,
            (
                SELECT COUNT(*)
                FROM (
                    SELECT * FROM `{destino}`
                    EXCEPT DISTINCT
                    SELECT * FROM `{origem}`
                )
            ) AS extras_destino
    """

    resultado = consultar_resultado(consulta)

    return (
        resultado.total_origem == resultado.total_destino
        and resultado.ausentes_destino == 0
        and resultado.extras_destino == 0
    )


def preparar_reconstrucao():
    planos = []

    for fonte, config in tabelas_reconstrucao.items():
        origem = config["origem"]
        destino = config["destino"]

        tabela_origem = client.get_table(origem)
        total_historico = validar_tabela_historica(
            fonte, config
        )

        try:
            tabela_destino = client.get_table(destino)
        except NotFound:
            tabela_destino = None

        total_atual = None

        if tabela_destino is not None:
            total_atual = contar_linhas(destino)

            schema_origem = [
                (campo.name, campo.field_type, campo.mode)
                for campo in tabela_origem.schema
            ]

            schema_destino = [
                (campo.name, campo.field_type, campo.mode)
                for campo in tabela_destino.schema
            ]

            if schema_origem != schema_destino:
                raise RuntimeError(
                    f"{fonte}: esquema incompatível "
                    "com a Bronze atual"
                )

        planos.append({
            "fonte": fonte,
            "origem": origem,
            "destino": destino,
            "total": total_historico,
            "total_atual": total_atual,
        })

        print(
            f"{fonte.upper()} | Histórico: {total_historico} | "
            f"Bronze atual: {total_atual}"
        )

    return planos


def reconstruir_bronze():
    planos = preparar_reconstrucao()

    if not EXECUTAR_RECONSTRUCAO:
        print()
        print("Simulação aprovada.")
        print("Nenhuma tabela oficial foi modificada.")
        return

    identificador = datetime.now(
        timezone.utc
    ).strftime("%Y%m%d_%H%M%S_%f")

    # Criar e conferir todos os backups antes da promoção.
    for plano in planos:
        if plano["total_atual"] is None:
            continue

        backup = (
            f"{PROJETO}.bronze.backup_reconstrucao_"
            f"{plano['fonte']}_{identificador}"
        )

        job = client.copy_table(
            sources=plano["destino"],
            destination=backup,
            job_config=bigquery.CopyJobConfig(
                write_disposition="WRITE_EMPTY"
            ),
            location="southamerica-east1",
        )
        job.result()

        if not comparar_tabelas(
            plano["destino"], backup
        ):
            raise RuntimeError(
                f"{plano['fonte']}: backup inconsistente"
            )

        print(
            f"{plano['fonte'].upper()} | "
            f"Backup validado: {backup}"
        )

    for plano in planos:
        job = client.copy_table(
            sources=plano["origem"],
            destination=plano["destino"],
            job_config=bigquery.CopyJobConfig(
                write_disposition="WRITE_TRUNCATE"
            ),
            location="southamerica-east1",
        )
        job.result()

        if not comparar_tabelas(
            plano["origem"], plano["destino"]
        ):
            raise RuntimeError(
                f"{plano['fonte']}: falha após reconstrução"
            )

        print(
            f"{plano['fonte'].upper()} | "
            f"Bronze reconstruída: {plano['total']} registros"
        )

    print()
    print("Reconstrução da Bronze concluída.")
    print("Silver e Gold precisam ser atualizadas.")


reconstruir_bronze()

SELIC | Histórico: 1052 | Bronze atual: 1052
CAMBIO | Histórico: 723 | Bronze atual: 723
IPCA | Histórico: 36 | Bronze atual: 36
SELIC | Backup validado: macrodata-brasil.bronze.backup_reconstrucao_selic_20261117_150903_769002
CAMBIO | Backup validado: macrodata-brasil.bronze.backup_reconstrucao_cambio_20261117_150903_769002
IPCA | Backup validado: macrodata-brasil.bronze.backup_reconstrucao_ipca_20261117_150903_769002
SELIC | Bronze reconstruída: 1052 registros
CAMBIO | Bronze reconstruída: 723 registros
IPCA | Bronze reconstruída: 36 registros

Reconstrução da Bronze concluída.
Silver e Gold precisam ser atualizadas.
